In [0]:
import pandas as pd
import yfinance as yf
from datetime import datetime
from dateutil.relativedelta import relativedelta

tickers = spark.table("workspace.default.ticker").toPandas()["ticker"].to_list()
today = datetime.today()
begining_of_the_year    = pd.to_datetime(datetime(today.year, 1, 1))
begining_of_the_quarter = pd.to_datetime(datetime(today.year, 3*((today.month-1)//3)+1, 1))
begining_of_the_month   = pd.to_datetime(datetime(today.year, today.month, 1))
start = datetime.today() - relativedelta(years=5,days=5)
ohlcv = yf.download(tickers, start=start, end=today, threads=False, auto_adjust=False)
ohlcv.index = pd.to_datetime(ohlcv.index.date)
ohlcv["Adj Close"]

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")
spark.createDataFrame(ohlcv["Adj Close"].reset_index()) \
        .write \
        .format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable("workspace.bronze.ohlcv_adj_close") 